# CardioSentry hybrids — B. Everything round A could not settle

Four runs in one session: **CBAM at 768**, **FPN at 768**, the
**dual-resolution** hybrid, and the **heterogeneous** dual.

Four independent sigmoid heads: **STEMI, AF, LVH, NORMAL**.
This round exists to improve **LVH**, which in round 1 accumulated 2.20 errors
per positive case against 1.11 for AF — the clearest finding in that run, and
unanimous across all five backbones.

**Run the cells in order, top to bottom.**
**Before starting: Runtime → Change runtime type.**

**Measured**, identical code and micro-batch, on `resnet50_base` epoch 2:

| runtime | vCPU | steady rate | whole round | units/hr | **total units** |
|---|---:|---:|---:|---:|---:|
| T4 standard | 2 | 25 img/s | ~11 h | 1.0 | ~11 |
| T4 high-RAM | 8 | 84.5 img/s | ~8.0 h | 1.24 | ~10.0 |
| **L4** | 12 | **149.7 img/s** | **~4.5 h** | 1.54 | **~7.0** |

**L4 is the cheaper option, not the dearer one.** It is 1.77x faster for 1.24x
the hourly rate, so it uses ~30% *fewer* compute units and half the wall clock.
Hourly rate is the wrong number to optimise — you rent by the hour and need
fewer of them.

The 22 GB matters too: it protects the uniform micro-batch of 8 (cell 5), which
is what keeps BatchNorm comparable across runs. On a 14.6 GB T4 the heaviest
models are tight at micro 8, and an OOM there forces *every* run down to 4.

---
*The whole harness (af_lvh.py, artifacts.py, baseline.py, compare.py, config.py, data.py, engine.py, hybrids.py, manifest.py, metrics.py, preflight.py, prepare_dataset.py, subgroup.py, train.py, requirements.txt, run_hybrids.sh) is embedded in cell 1 — no folder upload needed.*

## What notebook A settled, and the one flaw in how it settled it

A ran the control: `resnet50_base` and `resnet50_hires` are the same model, same
head, same protocol, same files, same micro-batch — **only the input resolution
differs.** 768 won, and beat both attention hybrids.

| model | views | pooled LVH AUPRC | **PTB-XL LVH AUPRC** |
|---|---:|---:|---:|
| **`resnet50_hires`** | **768** | **0.9248** | **0.9315** |
| `resnet50_base` | 384 | 0.9036 | 0.9270 |
| `resnet50_cbam` | 384 | 0.8901 | 0.9097 |
| `resnet50_fpn` | 384 | 0.8767 | 0.8976 |

**Now read the `views` column.** The only model that beat the 384 baseline is the
only model that was given more pixels. CBAM and FPN were never run at 768, so
that table compares an *architecture* change against a *resolution* change and
concludes the architectures were worthless. **That conclusion is not available
from that experiment**, and half this notebook exists to fix it.

Two findings from A do carry in unchanged:

**Most of the pooled 768 gain was not about LVH.** Stratified by database,
+0.0212 pooled becomes **+0.0045 inside PTB-XL** (95% CI −0.0124 … +0.0208),
where 234 of the 255 LVH positives live. The rest came from the 768 model
learning to stop calling LVH on STEMI-database images — 11 false positives → 1
— which is database recognition, not electrocardiography. There *is* a genuine
residue: inside PTB-XL, 768 missed 15 fewer LVH at essentially unchanged FPR
(sensitivity 0.786 → 0.850). Real, but about a third the headline size.
**Cell 7c is the primary read of this notebook.**

**768 did not cost the other heads.** No AUPRC or sensitivity regression on
STEMI, AF or NORMAL. NORMAL is the only head trending down (−0.017 inside
PTB-XL, not significant) and it is the rule-out head, so watch it.

## The four runs, in the order they run

**1 · `resnet50_dual` (384+768) — the headline hybrid.** One ResNet-50 on the
whole sheet at 384, a second on the same sheet at 768, fused by concatenation.
*Was higher-resolution preprocessing sufficient, making the dual-branch
complexity unnecessary?* It runs first because it is the round's central
question and because run 4 is uninterpretable without it.

**2 · `resnet50_cbam_hires` (768) — does attention become useful once
fine-grained information is preserved?** At 384 the sheet is **1.73 px/mm**
vertically, so a 0.5 mV LVH voltage difference — what Sokolow-Lyon measures —
spans **under 9 pixels**. Attention cannot reweight detail the resampling already
destroyed. At 768 it is 3.46 px/mm.

**3 · `resnet50_fpn_hires` (768) — does multi-scale extraction benefit from
higher-resolution input?** An FPN's stride-4 level is only finer than its
stride-8 level if the pixels underneath differ. In round A, zeroing *any* pyramid
level moved macro AUPRC by **≤0.0016** — it had multi-scale structure and used
none of it. If that is because every level read the same blurred waveform, 768
should change it. If it ablates just as flat here, the FPN is inert on this task
at any resolution, and that is worth knowing once rather than suspecting forever.

**4 · `resnet50_effb3_dual` (384+768) — does a different CNN provide
complementary fine-detail features?** `resnet50_dual` fuses two identical
ResNet-50s, so whatever the 768 branch adds, it adds through the same inductive
biases. This swaps the detail branch for EfficientNet-B3.

Runs 2 and 3 each pair exactly with their 384 twin from notebook A — same
everything, **only the resample differs** — so cell 7d can print them as pairs and
separate "this architecture is worthless" from "this architecture was starved."

## ⚠️ Before interpreting run 4

In round 1 EfficientNet-B3 finished last on all nine headline metrics — but its
**epoch-1 validation AUPRC was 0.4467 against 0.77–0.83 for every other
backbone**. That is the signature of a learning rate mismatched to the
architecture, not of a weak architecture, and no LR sweep was ever run
(`sweep = false` in every round-1 `result.json`). Fusing a possibly-mis-optimised
branch under the same shared 1e-4 risks re-measuring that confound rather than
testing heterogeneous fusion.

**A poor result on run 4 is UNINTERPRETABLE** until an equal-budget sweep has run
(`--sweep`, the same three LRs for every model). A good result is still
informative — the confound can only hurt this model, not flatter it. It runs last
so a dead session costs you the least important run.

## The bar, and the column that decides

`resnet50_hires` reached **0.9315 PTB-XL LVH AUPRC** with 24.6 M parameters, one
forward pass and 37 minutes. That is the bar — **not** round 1's 0.8805, and not
`resnet50_base`. Cell 7d scores each rung against the rung below it.

**A PTB-XL gain under about 0.01 does not pay for 2× the parameters and a second
forward pass per image**, and the round's finding becomes *"higher-resolution
preprocessing was sufficient; the dual-branch complexity was unnecessary."* That
is a perfectly good result, cheaper to deploy and easier to defend than any
hybrid. Cell 7d states the threshold before the numbers arrive, so the call is
not made post hoc.

**Rank on the PTB-XL column, not the pooled one.** A model that merely learns to
recognise the STEMI database stops calling LVH on its 146 images and its pooled
score rises without it reading a single QRS complex.

**Cell 7d also runs the stream ablation.** `without_hi` zeroes the 768 feature
group and re-scores TEST with the same frozen thresholds. If zeroing `hi` barely
moves the score, the dual never used the extra pixels — it ensembled two ImageNet
backbones, and a second 384 backbone would have produced the same gain more
cheaply. Read the ablation *and* the ladder; when they disagree, the ablation
wins.

## ⏱️ This is a ~4.5 h session — how to survive a disconnect

`run_hybrids.sh` writes to Drive as it goes and **does not abort if one model
fails** — the rest still run. Every run resumes from its last completed epoch, and
a model that already finished is skipped, so re-running cell 5 after a
disconnect costs a minute rather than a night.

If your session will not last 4.5 h, **use the per-model cells below cell 5**
instead. They run one model at a time in the order above, and each line is
independently resumable.

Results land in `cardiosentry_runs/hybrid_v1/<model>/` — the same folder notebook
A wrote to. That is deliberate: `compare.py` reads a single directory and ranks
whatever it finds, so the shared folder is what lets one table put a 768 CBAM run
next to a dual-resolution run. Organisation comes from the per-model
subdirectory names.

## Cell 1 — unpack the code and install timm

In [ ]:
import base64, io, tarfile, os
os.makedirs("/content/code", exist_ok=True)
_PKG = "..."
tarfile.open(fileobj=io.BytesIO(base64.b64decode(_PKG)), mode="r:gz").extractall("/content/code")
os.chmod("/content/code/run_hybrids.sh", 0o755)
print("unpacked:")
for f in sorted(os.listdir("/content/code")):
    print("   ", f)

!pip install -q timm
!nvidia-smi --query-gpu=name,memory.total --format=csv

# What this runtime actually is. vCPU count is the number that matters most:
# the 768px corpus is JPEG-decode-bound, and decode scales with cores, not RAM.
import os, shutil
_cores = os.cpu_count()
_ram = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1024**3
try:
    _disk = shutil.disk_usage("/content").free / 1024**3
except OSError:
    _disk = float("nan")
print(f"\nvCPU {_cores}   RAM {_ram:.0f} GB   free disk {_disk:.0f} GB")
print(f"loader will use {min(_cores, 8)} worker(s)")
if _cores <= 2:
    print("  2 vCPU -> decode-bound at ~27 img/s. On Colab Pro switch to L4:")
    print("  Runtime > Change runtime type > L4. Same compute units for this")
    print("  round, under half the wall clock.")
else:
    print(f"  projected decode ceiling ~{27 * min(_cores, 8) / 2:.0f} img/s "
          f"(the 2-vCPU T4 baseline was 27)")
print("  RAM is NOT the bottleneck - peak usage here is about 5 GB. Turn on")
print("  High-RAM only if it raises the vCPU count on your runtime; if it")
print("  grants memory alone it costs compute units and buys nothing.")

## Cell 2 — mount Drive

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

# Cache pretrained weights on Drive (~1.4 GB, once). Without this every runtime
# reset re-downloads them from HuggingFace - slow, and it is what triggers the
# "unauthenticated requests" rate limit over a night of reconnects.
os.environ["HF_HOME"] = "/content/drive/MyDrive/CardioSentry/hf_cache"

# HuggingFace token - raises Hub rate limits. Stored in Colab Secrets, never in
# this notebook: a pasted token gets saved into the .ipynb and travels with any
# copy you share.
try:
    from google.colab import userdata
    _tok = userdata.get("HF_TOKEN")
    if _tok:
        os.environ["HF_TOKEN"] = _tok
        os.environ["HUGGING_FACE_HUB_TOKEN"] = _tok
        print("HF token loaded from Colab Secrets")
    else:
        print("HF_TOKEN secret is empty - continuing unauthenticated (works fine)")
except Exception as exc:
    print(f"no HF_TOKEN in Colab Secrets ({type(exc).__name__}) - "
          f"continuing unauthenticated (works fine)")

# ---- Drive layout. Edit DRIVE_ROOT only; everything else follows. ----------
DRIVE_ROOT = "/content/drive/MyDrive/CardioSentry"
RUN_NAME   = "hybrid_v1"

DATASETS = f"{DRIVE_ROOT}/cardiosentry_dataset"
SRC_TAR  = f"{DATASETS}/prototype_1152.tar"      # the original 1686x1311 corpus
TAR_768  = f"{DATASETS}/prototype_1152_768.tar"  # built once by notebook 0
RUNS     = f"{DRIVE_ROOT}/cardiosentry_runs/{RUN_NAME}"

# ALWAYS local. Training off the mounted Drive turns every image read into a
# network round trip and starves the GPU - /content is wiped on each runtime
# reset, which is why the notebooks re-extract (about a minute) every session.
DATA = "/content/prototype_1152_768"

os.makedirs(RUNS, exist_ok=True)
for _p, _label in ((SRC_TAR, "source corpus"), (TAR_768, "768px corpus")):
    print(f"  {'FOUND ' if os.path.exists(_p) else 'missing'}  {_label:<14} {_p}")
print("\nDATA =", DATA, "\nRUNS =", RUNS)

## Cell 3 — extract the 768px dataset (~1 min)

Built once by notebook **0_Prepare**. This cell refuses to fall back to round 1's
384px folder, because feeding upsampled data to the high-resolution models would
invalidate the comparison silently.

In [ ]:
# Extract the 768px copy built by notebook 0.
#
# Every step is checked. A failed shell command does NOT stop a Colab cell, so
# an unchecked `!tar -xf` prints a traceback into the void and the cell sails on
# to report success - which is exactly how you end up staring at a missing
# index.csv with no idea which step actually failed.
import os, subprocess, tarfile

EXPECTED_FILES = 6670          # images; index.csv makes 6671


def _images_present(root):
    if not os.path.isdir(root):
        return 0
    return sum(len([f for f in files if f.endswith(".jpg")])
               for _, _, files in os.walk(root))


def _have_dataset(root):
    return os.path.exists(os.path.join(root, "index.csv")) and            _images_present(root) >= EXPECTED_FILES


if _have_dataset(DATA):
    print("already extracted:", DATA)
elif not os.path.exists(TAR_768):
    raise SystemExit(
        f"MISSING: {TAR_768}\n"
        f"Run notebook 0_Prepare first - it builds this once and tars it to Drive.")
else:
    gb = os.path.getsize(TAR_768) / 1024**3
    print(f"extracting {TAR_768}  ({gb:.2f} GB)")

    # Verify the archive BEFORE trusting it. Writing ~2 GB straight to the Drive
    # FUSE mount truncates more often than anyone expects, especially if the
    # notebook-0 session ended before Drive finished flushing. prepare_dataset.py
    # copies index.csv LAST, so it sits near the end of the archive - a truncated
    # tar loses precisely that file and nothing else, which looks like a mystery
    # rather than a broken download.
    try:
        with tarfile.open(TAR_768) as _t:
            _names = _t.getnames()
        print(f"  archive intact: {len(_names)} entries")
        _has_index = any(n.endswith("index.csv") for n in _names)
    except Exception as exc:
        raise SystemExit(
            f"\nTHE TAR ON DRIVE IS TRUNCATED OR CORRUPT: {type(exc).__name__}: {exc}\n"
            f"  size on Drive: {gb:.2f} GB\n"
            f"  It never finished writing. Delete it and re-run notebook 0:\n"
            f"      !rm -f '{TAR_768}'\n"
            f"  Notebook 0 now builds the tar on local disk and verifies it\n"
            f"  before copying it to Drive, so this will not recur.")

    r = subprocess.run(["tar", "-xf", TAR_768, "-C", "/content/"],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise SystemExit(f"tar failed (exit {r.returncode}):\n{r.stderr[:2000]}")
    print("  extracted ->", DATA)

    # index.csv is copied verbatim by prepare_dataset.py and rel_path is unchanged
    # by design, so the copy in the ORIGINAL tar is byte-identical. If only that
    # file is missing there is no need to rebuild 6,670 images to recover it.
    if not _has_index and _images_present(DATA) >= EXPECTED_FILES:
        print("  index.csv missing from the archive but the images are all here")
        print("  recovering it from the original tar (it is copied verbatim by")
        print("  prepare_dataset.py, and rel_path is unchanged by design)")
        import shutil, tempfile
        with tarfile.open(SRC_TAR) as _st:
            _member = next((m for m in _st.getnames()
                            if m.endswith("index.csv")), None)
            if _member is None:
                raise SystemExit(f"no index.csv inside {SRC_TAR} either - "
                                 f"re-run notebook 0 from the original corpus.")
            with tempfile.TemporaryDirectory() as _tmp:
                _st.extract(_member, _tmp)
                shutil.copy2(os.path.join(_tmp, _member),
                             os.path.join(DATA, "index.csv"))
        print(f"  recovered from {_member}")

n_img = _images_present(DATA)
print(f"\nimages on disk: {n_img:,}")
if not os.path.exists(os.path.join(DATA, "index.csv")):
    raise SystemExit(
        f"no index.csv at {DATA} even after extraction.\n"
        f"images found: {n_img:,} of {EXPECTED_FILES:,}\n"
        f"If the image count is right, only index.csv is missing - recover it with:\n"
        f"    !tar -xf '{SRC_TAR}' -C /content/ prototype_1152/index.csv\n"
        f"    !cp /content/prototype_1152/index.csv '{DATA}/index.csv'\n"
        f"Otherwise delete the tar on Drive and re-run notebook 0.")
if n_img < EXPECTED_FILES:
    print(f"WARNING: expected {EXPECTED_FILES:,} images, found {n_img:,}. "
          f"The archive was probably truncated - preflight will fail the "
          f"integrity check. Re-run notebook 0.")

from PIL import Image
import pandas as pd
_df = pd.read_csv(os.path.join(DATA, "index.csv"))
print(f"index.csv: {len(_df):,} rows")
with Image.open(os.path.join(DATA, _df.rel_path.iloc[0])) as _im:
    print("stored image size:", _im.size)
    assert min(_im.size) >= 768, (
        "images are %s - this is the round-1 384px copy, not the 768px one. "
        "The resolution experiment would measure nothing." % (_im.size,))
print("\ndataset OK")
!df -h /content/drive/MyDrive | tail -1

## Cell 4 — preflight for these models only (~2 min)

In [ ]:
%cd /content/code
!python preflight.py --data "$DATA" --model resnet50_dual resnet50_cbam_hires resnet50_fpn_hires resnet50_effb3_dual     --micro-batch 8 --parallel 1

## Cell 5 — train  ⏳ THE LONG ONE (~~3.3 h on a 12-vCPU L4 high-RAM)

Writes to Drive as it goes, so a disconnect does not lose finished work, and
re-running resumes each model from its last completed epoch.

`run_hybrids.sh` does **not** abort if one model fails — the rest still run.

In [ ]:
%cd /content/code

# MICRO_BATCH applies the SAME value to every run, and that matters more than it
# sounds. BatchNorm normalises over the MICRO-batch, and gradient accumulation
# does not touch it - so runs at different micro-batches differ in BN statistics
# on top of whatever they were meant to differ in. On the T4 defaults
# resnet50_base runs at 16 and resnet50_hires at 4, which means the control pair
# is NOT "only the resolution differs".
#
#   24 GB card (L4 / A100) -> "8", and the comparison is clean
#   16 GB card (T4)        -> "4" fits everything but is slower; or leave it
#                             unset for per-model defaults and DECLARE the confound
import os
os.environ["MICRO_BATCH"] = "8"

# PARALLEL runs N models AT ONCE on the one GPU. LEAVE IT AT 1 ON AN L4.
#
# It was tried at 2 and MEASURED SLOWER, on this exact card and corpus:
#
#     model                PARALLEL=2      alone
#     resnet50_dual        19.5 img/s      ~34 img/s
#     resnet50_cbam_hires  11.7 img/s      ~30 img/s
#     aggregate            31.2 img/s
#
# Two models together move fewer images per second than resnet50_dual moves
# on its own. For that group it was ~137 min against ~95 min sequential.
#
# WHY, so nobody re-derives the wrong answer: an L4 at ~6 TFLOP/s against a
# ~121 TFLOP/s fp16 peak LOOKS like it has idle SMs to fill. It does not.
# 768px convolutions move huge activation tensors and the card has only
# ~300 GB/s of memory bandwidth, so the workload is BANDWIDTH-bound, not
# occupancy-bound, and a second process buys nothing while costing cache
# locality. The giveaway is in round A's own numbers: 159.4 -> 43.7 img/s
# from 384 to 768 is 3.65x for exactly 4x the pixels. Near-linear scaling
# with pixel count means the device is already saturated; idle SMs would
# have shown SUBLINEAR scaling.
#
# It also corrupts the COST table. train_seconds and the end-of-run
# inference benchmark both run under contention - and the benchmark runs
# after training, while the other model in the group is still going. That
# is half the "was the second branch worth it" decision, measured wrong.
#
# Accuracy is unaffected either way: same seeds, same micro-batch, same
# batch order, same arithmetic. Only the clock changes.
#
# Worth retrying only on a bandwidth-rich card (A100 ~2 TB/s, H100 ~3 TB/s).
# If a group OOMs there, lower PARALLEL - NEVER MICRO_BATCH, which would
# change BatchNorm statistics and break comparability with every finished run.
os.environ["PARALLEL"] = "1"

!./run_hybrids.sh "$DATA" "$RUNS" resnet50_dual resnet50_cbam_hires resnet50_fpn_hires resnet50_effb3_dual

### Prefer one model at a time?

Use these instead of cell 5. Re-running a line after a disconnect resumes it.
If you hit an OOM, add `--micro-batch 2` — gradient accumulation keeps the
effective batch at 32, so the run stays comparable.

In [ ]:
# !python train.py --model resnet50_dual        --data "$DATA" --out "$RUNS"
# !python train.py --model resnet50_cbam_hires  --data "$DATA" --out "$RUNS"
# !python train.py --model resnet50_fpn_hires   --data "$DATA" --out "$RUNS"
# !python train.py --model resnet50_effb3_dual  --data "$DATA" --out "$RUNS"

## Cell 6 — progress check

In [ ]:
!cat "$RUNS/_SUMMARY.txt"

In [ ]:
# did anything fail?
!for f in "$RUNS"/*/STATUS.txt; do echo "--- $f"; head -2 "$f"; done
!ls "$RUNS"/*/ERROR.txt 2>/dev/null && echo "^ failures above" || echo "no failures"

## Cell 7 — what these runs said about LVH

Sorted by LVH AUPRC, with deltas against round 1's ResNet-50. Works on a
half-finished batch.

In [ ]:
%cd /content/code
!python compare.py --runs "$RUNS" --sort lvh_auprc --csv "$RUNS/comparison.csv" | head -60

## Cell 7b — did anything else break?  ⚠️ read this before the headline

This round targets LVH. **Macro averages dilute a single-head regression by 4×**,
so a model that gains 0.04 on LVH and loses 0.04 on STEMI shows an unchanged
macro AUPRC and reads as a clean win in every headline table.

It is not one. An over-called LVH earns a follow-up echo; a missed STEMI does
not. Round 1's ResNet-50 missed **16 of 146 STEMIs** — that is the number a
"better" model has to not make worse.

In [ ]:
%cd /content/code
!python compare.py --runs "$RUNS" 2>/dev/null | sed -n "/PER-CONDITION REGRESSION/,/AF x LVH SUBGROUP/p" | head -50

## Cell 7c — LVH by source  🔴 THE PRIMARY READ, before any headline number

Notebook A is the reason this cell exists. Its pooled result looked clean —
768px lifted LVH AUPRC from 0.9036 to 0.9248, beating both attention hybrids.
Stratifying the test split by originating database took most of it away:

| | pooled | PTB-XL only |
|---|---:|---:|
| `resnet50_base` (384) | 0.9036 | 0.9270 |
| `resnet50_hires` (768) | 0.9248 | **0.9315** |
| delta | **+0.0212** | **+0.0045**  (95% CI −0.0124 … +0.0208) |

**PTB-XL carries 234 of the 255 LVH positives. Chapman carries 21. The STEMI
database carries none** — all 146 of its images are LVH-negative, so a model
that gets better at *recognising that database* stops calling LVH on them and
its pooled AUPRC rises without it having read a single QRS complex. That is
exactly what happened: LVH false positives on STEMI-database images went 11 → 1
from 384 to 768, while inside PTB-XL they went 30 → 28.

Worse, on this split **the STEMI label is *exactly* `source == "STEMI"`**, 146 of
146. Any STEMI metric here is a database classifier in disguise — which is why
`resnet50_hires` posted 0.9996 STEMI AUROC. Do not report that number.

**So: read the PTB-XL column. It is the one the shortcut cannot reach.**
`sens@.08` fixes specificity instead of trusting a globally-tuned threshold on a
subgroup, so it compares detectors rather than operating points.

In [ ]:
%cd /content/code
!python compare.py --runs "$RUNS" 2>/dev/null | sed -n "/BY SOURCE - is the gain/,/PER-CONDITION REGRESSION/p" | head -60

## Cell 7d — the resolution ladder, and what each branch was worth

Two questions, one after the other, and they have to be answered in this order.

**1. The ladder: 384 → 768 → both.** Each rung is scored against *the rung below
it*, on PTB-XL LVH AUPRC. Scoring `resnet50_dual` against round 1 measures three
changes at once and credits all of them to the second branch. The bar is
`resnet50_hires` at **0.9315** — 24.6 M params, one forward pass. A dual model
that gains less than about 0.01 over that has not paid for 2× the parameters and
a second pass per image, and the round's finding is *"higher-resolution
preprocessing was sufficient; the dual-branch complexity was unnecessary."*

**2. The ablation, which can overturn (1).** `without_hi` zeroes the 768 feature
group after its LayerNorm and re-scores TEST with the same frozen thresholds.
**If zeroing `hi` barely moves the score, the model never used the extra pixels**
— it ensembled two ImageNet backbones, and any gain is an ensembling gain that a
second 384 backbone would also have produced, more cheaply.

Notebook A's FPN is the cautionary case: zeroing *any* pyramid level moved macro
AUPRC by ≤0.0016. It had multi-scale structure and used none of it.

In [ ]:
%cd /content/code
!python compare.py --runs "$RUNS" 2>/dev/null | sed -n "/RESOLUTION LADDER/,/TEST SET/p"

## Cell 7e — AF × LVH subgroup: does LVH survive when AF is present?  🔴 THE QUESTION C ASKS

The AF × LVH subgroup analyses scored the frozen 768 reference and found it
detects AF correctly but **misses LVH when the two coexist**:

| head | recall on AF+LVH (n=28) | comparand | delta | Fisher p |
|---|---:|---:|---:|---:|
| LVH | **0.6429** (18/28) | 0.8722 on LVH-only (n=227) | −0.2294 | **0.0038** |
| AF | 0.9286 (26/28) | 0.9416 on AF-only (n=257) | −0.0131 | 0.5142 |

Verdict: **SUPPORTED** — 10 of the 28 AF+LVH cases were "AF correct + LVH
missed" against 2 the other way (McNemar p=0.0386).

Every run in this round is scored the same way, from its **own frozen
decisions**, so this table answers the question notebook C exists for: does
`resnet50_vit` (768+768) raise LVH recall on AF+LVH from 0.6429 without
dropping AF recall below 0.9286? AUPRC columns use the reference-pool design
(subgroup positives vs every true_<head> == 0 ECG) — small-sample reads, not
verdict reads. The subgroup has 28 ECGs, so the Fisher p and bootstrap CI say
how much of any delta is real.

In [ ]:
%cd /content/code
!python compare.py --runs "$RUNS" 2>/dev/null | sed -n "/AF x LVH SUBGROUP/,/THE CONTROL/p"

## Cell 8 — full report for one model

In [ ]:
MODEL = "resnet50_dual"          # change and re-run
print(open(f"{RUNS}/{MODEL}/report.txt").read())

## Cell 9 — LVH learning curves

The per-epoch LVH AUPRC is tracked separately from the macro monitor on purpose.
A model that improves macro AUPRC while flattening or degrading LVH has traded
the head you care about for the three you already had.

In [ ]:
import json, glob
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 3, figsize=(17, 4))
for f in sorted(glob.glob(f"{RUNS}/*/result.json")):
    r = json.load(open(f))
    h = r["history"]
    if not h:
        continue
    ep = [x["epoch"] for x in h]
    lab = f"{r['model']} ({'+'.join(map(str, r['model_info']['views']))})"
    ax[0].plot(ep, [x["train_loss"] for x in h], label=lab)
    ax[1].plot(ep, [x["val_macro_auprc"] for x in h], label=lab)
    ax[2].plot(ep, [x.get("val_LVH_auprc") for x in h], label=lab)
for a, t in zip(ax, ["train loss", "val macro AUPRC", "val LVH AUPRC"]):
    a.set_title(t); a.set_xlabel("epoch"); a.legend(fontsize=7); a.grid(alpha=.3)
# TWO reference lines, because they say different things. Round 1 is where this
# work started; resnet50_hires is what a new model actually has to beat. Drawing
# only the first one makes every model in this notebook look like a triumph.
ax[2].axhline(0.8805, ls=":", c="0.5", lw=1)
ax[2].annotate("round 1 resnet50 (test)", (0.02, 0.8805), xycoords=("axes fraction", "data"),
               fontsize=7, va="bottom", color="0.4")
ax[2].axhline(0.9248, ls="--", c="crimson", lw=1.2)
ax[2].annotate("notebook A resnet50_hires 768 (test) - THE BAR", (0.02, 0.9248),
               xycoords=("axes fraction", "data"), fontsize=7, va="bottom", color="crimson")
plt.tight_layout(); plt.show()

# ...and every head separately. LVH is the target, but STEMI is the one whose
# regression matters clinically, so it gets its own panel rather than being
# averaged into the macro line above.
R1 = {"STEMI": 0.9560, "AF": 0.9777, "LVH": 0.8805, "NORMAL": 0.9534}
# Notebook A's single-stream 768 run, on the same frozen test split. This is the
# row a dual-resolution model has to beat, head by head - not round 1.
HIRES = {"STEMI": 0.9979, "AF": 0.9825, "LVH": 0.9248, "NORMAL": 0.9523}
fig, ax = plt.subplots(1, 4, figsize=(20, 3.6), sharey=True)
for a, lab in zip(ax, ["STEMI", "AF", "LVH", "NORMAL"]):
    for f in sorted(glob.glob(f"{RUNS}/*/result.json")):
        r = json.load(open(f))
        h = r["history"]
        if not h:
            continue
        a.plot([x["epoch"] for x in h],
               [x.get(f"val_{lab}_auprc") for x in h], label=r["model"])
    a.axhline(R1[lab], ls=":", c="0.5", lw=1)
    a.axhline(HIRES[lab], ls="--", c="crimson", lw=1.2)
    a.set_title(f"val {lab} AUPRC\n(dotted = round-1 resnet50, dashed = A's hires 768)",
                fontsize=8)
    a.set_xlabel("epoch"); a.grid(alpha=.3)
ax[0].legend(fontsize=7)
plt.tight_layout(); plt.show()

# STEMI's panel is the one to distrust, not to celebrate. Its label is exactly
# "source == STEMI" on this split, so the crimson line at 0.9979 is what perfect
# database recognition scores - not what reading infarcts scores. A model that
# reaches it has learned the corpus. See cell 7c.
print("STEMI panel: label == source on this split. 0.9979 is the database-"
      "recognition ceiling,\n             not a clinical result. Do not report it.")